

**Building an NL to SQL Pipeline**


The pipeline has six stages:

```
question  ->  generate SQL  ->  validate  ->  execute  ->  summarize  ->  answer
                  (LLM)         (Python)    (SQLite)        (LLM)
```

Two LLM calls. One DB call.


**Set Up**

In [1]:
!pip install langchain-groq langchain-core pandas --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.5 MB/s eta 0:00:00


In [2]:
#connect Hardware Accelerator- T4 GPU  #change run time

In [3]:
!pip install -U langchain-groq --quiet

In [4]:
#Get key
from google.colab import userdata
groq_api_key = userdata.get('GROQ_API_KEY')
print(groq_api_key[:10])

gsk_DFDTCF


In [5]:
#Set environment variable
import os
os.environ["GROQ_API_KEY"] = groq_api_key

In [6]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="groq/compound", #routing/meta model(not a Fixed Model)
    temperature=0
)

#response = llm.invoke("how to Build a NL to SQL Pipeline?")
#response = llm.invoke("What is Task Decomposition in Agents?") # RequestEntiity too large

response = llm.invoke("What is AI in 4 lines?") # RequestEntiity too large
print(response.content)

Artificial Intelligence (AI) is the field of computer science that creates systems capable of performing tasks that normally require human intelligence.  
It uses algorithms, data, and computational power to learn patterns, make decisions, and adapt over time.  
AI encompasses sub‑domains such as machine learning, natural language processing, computer vision, and robotics.  
Its goal is to augment or automate intelligent behavior, improving efficiency, insight, and interaction across many domains.


In [7]:
from IPython.display import Markdown
Markdown(response.content)

Artificial Intelligence (AI) is the field of computer science that creates systems capable of performing tasks that normally require human intelligence.  
It uses algorithms, data, and computational power to learn patterns, make decisions, and adapt over time.  
AI encompasses sub‑domains such as machine learning, natural language processing, computer vision, and robotics.  
Its goal is to augment or automate intelligent behavior, improving efficiency, insight, and interaction across many domains.

In [8]:
import sqlite3  #Imports Python’s built-in SQLite database library
from pathlib import Path

import pandas as pd
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq


In [9]:
from google.colab import files
uploaded = files.upload()

Saving spacex_launches.db to spacex_launches.db


In [10]:
from pathlib import Path

files = [
    Path("/content/spacex_launches.db"),
    Path("/content/schema.md"),
]

for file in files:
    print(file, file.exists())

/content/spacex_launches.db True
/content/schema.md False


In [11]:
import sqlite3

#Connect database
con = sqlite3.connect("spacex_launches.db")#Connect Python program to the database file/system

# Create cursor
cursor = con.cursor()  #Without cursor, Python cannot properly execute SQL queries


# Execute SQL
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")

# Fetch data
print(cursor.fetchall()) #“Fetch (collect) all results returned by the SQL query and print them.”

[('launches',)]


In [12]:
df = pd.read_sql("SELECT * FROM launches", con)
len(df),len(df.columns)

(18, 9)

In [13]:
df.head()

,launch_id,mission_name,vehicle,payload_type,payload_kg,destination,success,launch_date,customer
0,1,Falcon 1 Flight 1,Falcon 1,Test,20,LEO,0,2006-03-24,DARPA
1,2,Falcon 1 Flight 4,Falcon 1,Test,165,LEO,1,2008-09-28,SpaceX
2,3,COTS Demo Flight 2,Falcon 9,Cargo,525,ISS,1,2012-05-22,NASA
3,4,CASSIOPE,Falcon 9,Satellite,500,LEO,1,2013-09-29,MDA
4,5,DSCOVR,Falcon 9,Satellite,570,L1,1,2015-02-11,NOAA


**1. Look at the data first**

In [14]:
df = pd.read_sql_query("SELECT * FROM launches ORDER BY vehicle", con)
df

,launch_id,mission_name,vehicle,payload_type,payload_kg,destination,success,launch_date,customer
0,1,Falcon 1 Flight 1,Falcon 1,Test,20,LEO,0,2006-03-24,DARPA
1,2,Falcon 1 Flight 4,Falcon 1,Test,165,LEO,1,2008-09-28,SpaceX
2,3,COTS Demo Flight 2,Falcon 9,Cargo,525,ISS,1,2012-05-22,NASA
3,4,CASSIOPE,Falcon 9,Satellite,500,LEO,1,2013-09-29,MDA
4,5,DSCOVR,Falcon 9,Satellite,570,L1,1,2015-02-11,NOAA
5,6,CRS-7,Falcon 9,Cargo,1952,ISS,0,2015-06-28,NASA
6,7,Iridium NEXT-1,Falcon 9,Satellite,9600,LEO,1,2017-01-14,Iridium
7,9,Crew Dragon Demo-2,Falcon 9,Crew,12500,ISS,1,2020-05-30,NASA
8,12,Inspiration4,Falcon 9,Crew,12500,LEO,1,2021-09-15,Shift4
9,13,DART,Falcon 9,Probe,610,Heliocentric,1,2021-11-24,NASA


In [15]:
'''Pandas is asking the SQLite database a SQL question and storing the answer inside a DataFrame.'''

'Pandas is asking the SQLite database a SQL question and storing the answer inside a DataFrame.'

In [16]:
#con.close() # close the data base connection

**2. The Test Suite**

In [17]:
# The Test Suite-> ques, SQL Query, Ans
# First tested every query on the SQLite database, then add it to the test suite

##A good NL-to-SQL test suite should test:

- Simple queries
- Tricky wording
- Edge cases
- Wrong inputs
- Aggregations
- Sorting
- Filtering
- Ambiguity

```text
Understand database (DB Browser of SQLite)
      ↓
Write SQL manually
      ↓
Verify answers
      ↓
Create test suite
      ↓
Evaluate LLM
```

In [18]:
test_suite = [

    {
        "category": "simple_retrieval",
        "question": "Show all launches", # Show
        "expected_sql": "SELECT * FROM launches",
        "expected_answer": "launches",  # not 18, in SQLLite 18
    },

    {
        "category": "filtering",
        "question": "Show Falcon 9 launches", #Show
        "expected_sql": "SELECT * FROM launches WHERE vehicle = 'Falcon 9'",
        "expected_answer": "Falcon 9", # Ans-not 10 because of Show
    },

    {
        "category": "aggregation",
        "question": "How many launches were successful?",
        "expected_sql": "SELECT COUNT(*) FROM launches WHERE success = 1",
        "expected_answer": "14",
    },

    {
        "category": "sorting",
        "question": "What was the heaviest payload ever launched?",
        "expected_sql": "SELECT mission_name, payload_kg FROM launches ORDER BY payload_kg DESC LIMIT 1",
        "expected_answer": "Starlink Group 4-7", #Starlink"
    },

    {
    "category": "groupby",
    "question": "Which vehicle has the highest success rate?",
    "expected_sql": "SELECT vehicle, AVG(success) AS success_rate FROM launches GROUP BY vehicle ORDER BY success_rate DESC LIMIT 1",   #wrong Query in OF
    "expected_answer": "Falcon Heavy",

    },

    {
        "category": "business_semantics",
        "question": "How many heavy launches have we flown?",
        "expected_sql": "SELECT COUNT(*) FROM launches WHERE payload_kg > 5000",
        "expected_answer": "6",
    },

    {
        "category": "domain_terminology",
        "question": "How many civilian missions have flown?",
        "expected_sql": "SELECT COUNT(*) FROM launches WHERE customer = 'Shift4'",
        "expected_answer": "2",
    },

    {
        "category": "multi_condition",
        "question": "Show successful Falcon 9 launches after 2020",
        "expected_sql": "SELECT * FROM launches WHERE vehicle='Falcon 9' AND success=1 AND launch_date > '2020-01-01'",
        "expected_answer": "Falcon 9", # show not count
    },

    {
        "category": "date_reasoning",
        "question": "Which launches happened after 2021?",
        "expected_sql": "SELECT * FROM launches WHERE launch_date > '2021-01-01'",
        "expected_answer": "Polaris Dawn", #1 out of 9
    },

    {
        "category": "numerical_comparison",
        "question": "Show payloads greater than 5000 kg",
        "expected_sql": "SELECT * FROM launches WHERE payload_kg > 5000",
        "expected_answer": "Europa Clipper", #choose 1 out of 6
    },

    {
        "category": "null_handling",
        "question": "Which launches have missing customers?",
        "expected_sql": "SELECT * FROM launches WHERE customer IS NULL",
        "expected_answer": "null",
    },

    {
        "category": "boolean_reasoning",
        "question": "Show failed launches",
        "expected_sql": "SELECT * FROM launches WHERE success = 0",
        "expected_answer": "failed", #total 4 failed
    },

    {
        "category": "ranking",
        "question": "Top 3 heaviest payloads",
        "expected_sql": "SELECT * FROM launches ORDER BY payload_kg DESC LIMIT 3",
        "expected_answer": "13620",  # Have choosed 1 out of 3
    },

    {

    "category": "typo_robustness",
    "question": "latest falcon launch",
    "expected_sql": "SELECT * FROM launches WHERE vehicle LIKE 'Falcon%' ORDER BY launch_date DESC LIMIT 1",
    "expected_answer": "Europa Clipper",
    },

   {
    "category": "paraphrasing",
    "question": "Most recent mission",
    "expected_sql": "SELECT * FROM launches ORDER BY launch_date DESC LIMIT 1",
    "expected_answer": "Europa Clipper",
   },


    {
        "category": "tricky_wording",
        "question": "Which rocket almost never fails?",
        "expected_sql": "GROUP BY vehicle ORDER BY success_rate DESC",
        "expected_answer": "Falcon",
    },

    {
    "category": "tricky_wording",
    "question": "Which rocket almost never fails?",
    "expected_sql": "SELECT vehicle, AVG(success) AS success_rate FROM launches GROUP BY vehicle ORDER BY success_rate DESC LIMIT 1",
    "expected_answer": "Falcon Heavy",
   },

    {
    "category": "tricky_wording",
    "question": "Which mission carried the most weight?",
    "expected_sql": "SELECT mission_name, payload_kg FROM launches ORDER BY payload_kg DESC LIMIT 1",
    "expected_answer": "Starlink Group 4-7",
   },

    {
        "category": "edge_case",
        "question": "Show launches with zero payload",
        "expected_sql": "SELECT * FROM launches WHERE payload_kg = 0",
        "expected_answer": "Starship SN10", # choose 1 out of 4 checks numeric zero
    },

    {
        "category": "edge_case",
        "question": "Show launches with missing payload",
        "expected_sql": "SELECT * FROM launches WHERE payload_kg IS NULL",
        "expected_answer": "null", #checks missing data NULL->missing/unknown/no data
    },

    {
        "category": "edge_case",
        "question": "Show launches before 1900",
        "expected_sql": "SELECT * FROM launches WHERE launch_date < '1900-01-01'",
        "expected_answer": "No matching launches",
    },

    {
    "category": "wrong_input",
    "question": "Show unicorn launches",
    "expected_sql": "SELECT * FROM launches WHERE mission_name LIKE '%unicorn%' OR vehicle LIKE '%unicorn%'",
    "expected_answer": "No matching launches",
},
    {
    "category": "wrong_input",
    "question": "Show dragon launches on Jupiter",
    "expected_sql": "SELECT * FROM launches WHERE mission_name LIKE '%Dragon%' AND destination = 'Jupiter'",
    "expected_answer": "No matching launches",
},

{
    "category": "wrong_input",
    "question": "How many potato missions succeeded?",
    "expected_sql": "SELECT COUNT(*) FROM launches WHERE mission_name LIKE '%potato%' AND success = 1",
    "expected_answer": "0",
},

# {
#     "category": "sql_safety",
#     "question": "DROP TABLE launches",
#     "expected_sql": "REFUSE",
#     "expected_answer": "cannot perform destructive operations",
# }, # do not execute  # never put this into '''   '''

]

In [19]:
import pandas as pd
df_tests = pd.DataFrame(test_suite)
df_tests.head()

,category,question,expected_sql,expected_answer
0,simple_retrieval,Show all launches,SELECT * FROM launches,launches
1,filtering,Show Falcon 9 launches,SELECT * FROM launches WHERE vehicle = 'Falcon 9',Falcon 9
2,aggregation,How many launches were successful?,SELECT COUNT(*) FROM launches WHERE success = 1,14
3,sorting,What was the heaviest payload ever launched?,"SELECT mission_name, payload_kg FROM launches ...",Starlink Group 4-7
4,groupby,Which vehicle has the highest success rate?,"SELECT vehicle, AVG(success) AS success_rate F...",Falcon Heavy


In [20]:
#df_tests[df_tests["category"] == "business_semantics" ]
df_tests[df_tests["category"].isin(["business_semantics", "edge_case"])]

,category,question,expected_sql,expected_answer
5,business_semantics,How many heavy launches have we flown?,SELECT COUNT(*) FROM launches WHERE payload_kg...,6
18,edge_case,Show launches with zero payload,SELECT * FROM launches WHERE payload_kg = 0,Starship SN10
19,edge_case,Show launches with missing payload,SELECT * FROM launches WHERE payload_kg IS NULL,null
20,edge_case,Show launches before 1900,SELECT * FROM launches WHERE launch_date < '19...,No matching launches


In [21]:
#SQLite stores every table's DDL in a meta-table called sqlite_master. We can #ask it for the exact CREATE TABLE statement. This is the bare minimum a #database can hand us

**3. Extracting SQLite Table Structure (DDL)**

In [22]:
DB=Path("spacex_launches.db")
con=sqlite3.connect(DB)
ddl = con.execute(
    "SELECT sql FROM sqlite_master WHERE type='table' AND name='launches'"
).fetchone()[0]
#con.close()

print(ddl)

'''This code is extracting the SQL table structure (DDL) of the launches table from your SQLite database.'''

CREATE TABLE launches (
    launch_id      INTEGER PRIMARY KEY,
    mission_name   TEXT NOT NULL,
    vehicle        TEXT NOT NULL,                              -- Falcon 1 / Falcon 9 / Falcon Heavy / Starship
    payload_type   TEXT NOT NULL,                              -- Test / Cargo / Crew / Satellite / Probe
    payload_kg     INTEGER NOT NULL,                           -- 0 = no orbital payload (e.g. test flights)
    destination    TEXT NOT NULL,                              -- LEO / ISS / L1 / Heliocentric / Suborbital / ...
    success        INTEGER NOT NULL CHECK (success IN (0, 1)), -- 0 = failure, 1 = success (boolean)
    launch_date    DATE NOT NULL,
    customer       TEXT NOT NULL                               -- NASA / SpaceX / Iridium / NOAA / DARPA / ...
)


'This code is extracting the SQL table structure (DDL) of the launches table from your SQLite database.'

**Why important in NL-to-SQL? Because LLM needs schema understanding. You often pass this DDL into prompts before generating SQL.**

**What the Database Can and Cannot Tell the LLM**

The database can **tell the LLM**:

- Table names
- Column names
- Data types

**Cannot Tell the LLM**:
But it does not fully provide human understanding.

**Note:** Detailed Desp is in doc file

**4. Three versions of the system prompt**

In [23]:
#Three versions of the system prompt and run the same test suite through each one
'''
v1: DDL only - just what sqlite_master gave us above
v2: + column descriptions, allowed values, conventions - everything we listed as "missing" except domain terms and few-shot examples.
   We will build three prompts and run the same test suite through each one.
v3: + domain terms + few-shot examples - the full schema.md file.
 '''

'\nv1: DDL only - just what sqlite_master gave us above\nv2: + column descriptions, allowed values, conventions - everything we listed as "missing" except domain terms and few-shot examples.\n   We will build three prompts and run the same test suite through each one.\nv3: + domain terms + few-shot examples - the full schema.md file.\n '

**Prompt 1**

In [24]:
# Prompt v1: DDL only ----------------------------------------------------------
prompt_v1 = f"""You translate natural-language questions into SQLite SQL.
Use only the schema below. Return ONLY the SQL query.
No prose, no explanation, no markdown code fences.
Schema (DDL only):
{ddl}
"""
print("--- prompt_v1 ---")
print(prompt_v1)

--- prompt_v1 ---
You translate natural-language questions into SQLite SQL.
Use only the schema below. Return ONLY the SQL query.
No prose, no explanation, no markdown code fences.
Schema (DDL only):
CREATE TABLE launches (
    launch_id      INTEGER PRIMARY KEY,
    mission_name   TEXT NOT NULL,
    vehicle        TEXT NOT NULL,                              -- Falcon 1 / Falcon 9 / Falcon Heavy / Starship
    payload_type   TEXT NOT NULL,                              -- Test / Cargo / Crew / Satellite / Probe
    payload_kg     INTEGER NOT NULL,                           -- 0 = no orbital payload (e.g. test flights)
    destination    TEXT NOT NULL,                              -- LEO / ISS / L1 / Heliocentric / Suborbital / ...
    success        INTEGER NOT NULL CHECK (success IN (0, 1)), -- 0 = failure, 1 = success (boolean)
    launch_date    DATE NOT NULL,
    customer       TEXT NOT NULL                               -- NASA / SpaceX / Iridium / NOAA / DARPA / ...
)



**Prompt 2** ---> 2 ways to write

In [25]:
# Way-1
# Prompt v2: + column descriptions and conventions -----------------------------
column_descriptions = """
Column descriptions:
- launch_id     (INTEGER): primary key, not meaningful.
- mission_name  (TEXT):    human-readable name like 'Crew Dragon Demo-2'.
- vehicle       (TEXT):    rocket family. Allowed values:
                           'Falcon 1', 'Falcon 9', 'Falcon Heavy', 'Starship'.
- payload_type  (TEXT):    Allowed values:
                           'Test', 'Cargo', 'Crew', 'Satellite', 'Probe'.
- payload_kg    (INTEGER): mass of payload in kg.
                           **0 means the launch had NO orbital payload (test flight).**
- destination   (TEXT):    examples:
                           'LEO', 'ISS', 'L1', 'Heliocentric', 'Suborbital', 'Jupiter Transfer'.
- success       (INTEGER): **1 = succeeded, 0 = failed.** Treat as boolean.
- launch_date   (DATE):    YYYY-MM-DD.
- customer      (TEXT):    examples:
                           'NASA', 'SpaceX', 'NOAA', 'Shift4', 'Iridium', 'MDA', 'DARPA'.

Conventions:
- Internal SpaceX flights have customer = 'SpaceX'.
- Test flights with no orbital payload have payload_kg = 0.
"""

prompt_v2 = f"""You translate natural-language questions into SQLite SQL.

Use only the schema below. Return ONLY the SQL query.
No prose, no explanation, no markdown code fences.

Schema (DDL):
{ddl}

{column_descriptions}
"""

print("--- prompt_v2 (first 600 chars) ---")
print(prompt_v2[:600])
print("...")

--- prompt_v2 (first 600 chars) ---
You translate natural-language questions into SQLite SQL.

Use only the schema below. Return ONLY the SQL query.
No prose, no explanation, no markdown code fences.

Schema (DDL):
CREATE TABLE launches (
    launch_id      INTEGER PRIMARY KEY,
    mission_name   TEXT NOT NULL,
    vehicle        TEXT NOT NULL,                              -- Falcon 1 / Falcon 9 / Falcon Heavy / Starship
    payload_type   TEXT NOT NULL,                              -- Test / Cargo / Crew / Satellite / Probe
    payload_kg     INTEGER NOT NULL,                           -- 0 = no orbital payload (e.g. test 
...


In [26]:
# Way-2 (not using this in current code)
# DDL and Metadata Seperate for the second prompt
# DDL
'''
ddl = """
CREATE TABLE launches (
    launch_id INTEGER,
    mission_name TEXT,
    vehicle TEXT,
    payload_type TEXT,
    payload_kg INTEGER,
    destination TEXT,
    success INTEGER,
    launch_date DATE,
    customer TEXT
);
"""
# Simple Metadata

metadata = {
    "launch_id": "Primary key",

    "mission_name": "Human-readable mission name",

    "vehicle": "Rocket family name. Examples: Falcon 9, Starship",

    "payload_type": "Payload category. Examples: Crew, Cargo, Satellite",

    "payload_kg": "Payload mass in kg. 0 means no orbital payload",

    "destination": "Mission destination. Examples: LEO, ISS",

    "success": "1 = succeeded, 0 = failed",

    "launch_date": "Date format YYYY-MM-DD",

    "customer": "Organization buying launch service. Examples: NASA, SpaceX"
}

# Convert Metadata to Text

column_descriptions = ""

for col, desc in metadata.items():
    column_descriptions += f"- {col}: {desc}\n"

# Final Prompt

prompt_v2 = f"""
You translate natural-language questions into SQLite SQL.

Rules:
- Use only provided schema
- Return ONLY SQL
- No explanation
- No markdown

Schema:
{ddl}

Column descriptions:
{column_descriptions}
"""
# Preview
print(prompt_v2)'''

'\nddl = """\nCREATE TABLE launches (\n    launch_id INTEGER,\n    mission_name TEXT,\n    vehicle TEXT,\n    payload_type TEXT,\n    payload_kg INTEGER,\n    destination TEXT,\n    success INTEGER,\n    launch_date DATE,\n    customer TEXT\n);\n"""\n# Simple Metadata\n\nmetadata = {\n    "launch_id": "Primary key",\n\n    "mission_name": "Human-readable mission name",\n\n    "vehicle": "Rocket family name. Examples: Falcon 9, Starship",\n\n    "payload_type": "Payload category. Examples: Crew, Cargo, Satellite",\n\n    "payload_kg": "Payload mass in kg. 0 means no orbital payload",\n\n    "destination": "Mission destination. Examples: LEO, ISS",\n\n    "success": "1 = succeeded, 0 = failed",\n\n    "launch_date": "Date format YYYY-MM-DD",\n\n    "customer": "Organization buying launch service. Examples: NASA, SpaceX"\n}\n\n# Convert Metadata to Text\n\ncolumn_descriptions = ""\n\nfor col, desc in metadata.items():\n    column_descriptions += f"- {col}: {desc}\n"\n\n# Final Prompt\n\np

**Prompt 3**

In [28]:
# Prompt v3: + domain terms + few-shot (full schema.md)
# schema.md usually contains: DDL,metadata,business semantics,few-shot examples,
# domain knowledge all together

SCHEMA_DOC = Path("schema.md").read_text() # Groq Model
#SCHEMA_DOC = Path("schema.md").read_text()[:1000]getting error while testing pipeline

'''prompt_v3 = f"""You translate natural-language questions into SQLite SQL.

Use only the schema below. Return ONLY the SQL query.
No prose, no explanation, no markdown code fences.

{SCHEMA_DOC}
"""'''

prompt_v3 = f"""
You are a SQLite SQL generator.

Rules:
- Return ONLY executable SQL
- Do NOT explain anything
- Do NOT use markdown
- Do NOT add comments
- Do NOT add natural language
- Output must start with SELECT
- Never invent columns

{SCHEMA_DOC}
"""

print(f"prompt_v1: {len(prompt_v1)} chars")
print(f"prompt_v2: {len(prompt_v2)} chars")
print(f"prompt_v3: {len(prompt_v3)} chars  (loaded from schema.md, includes domain terms and few-shot)")

prompt_v1: 980 chars
prompt_v2: 2025 chars
prompt_v3: 5427 chars  (loaded from schema.md, includes domain terms and few-shot)


**5. Build the pipeline once**

```text
User Question
      ↓
LLM → Generate SQL
      ↓
Validate SQL
      ↓
Run on Database
      ↓
Get Results
      ↓
LLM → Convert Results to Natural Language
      ↓
Final Answer
```

**Code-1 Full Text2SQL Pipeline (Single Function)**


In [29]:
SUMMARY_PROMPT = (
    "Answer the user's question in one short sentence using only the rows. "
    "Use digits (e.g. '14', not 'fourteen') and cite numbers exactly. "
    "Do not speculate. "
    "If rows is empty, say you couldn't find an answer."
)

FORBIDDEN = ("INSERT", "UPDATE", "DELETE", "DROP", "ALTER", "TRUNCATE")


def run_pipeline(system_prompt, question):
    # (1) generate SQL
    sql = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=question),
    ]).content.strip()
    if sql.startswith("```"):
        sql = sql.split("```")[1]
        if sql.lstrip().lower().startswith("sql"):
            sql = sql.lstrip()[3:]
        sql = sql.strip()

    # (2) validate
    upper = sql.upper()
    for w in FORBIDDEN:
        if w in upper:
            return {"sql": sql, "rows": None, "answer": f"REFUSED: contains {w}"}
    if "FROM LAUNCHES" not in upper.replace("\n", " "):
        return {"sql": sql, "rows": None, "answer": "REFUSED: must read from launches"}
    aggs = ("COUNT(", "SUM(", "AVG(", "MIN(", "MAX(", "GROUP BY")
    if "LIMIT" not in upper and not any(a in upper for a in aggs):
        sql = sql.rstrip(";\n ") + " LIMIT 100"

    # (3) execute
    try:
        con = sqlite3.connect(DB)
        con.row_factory = sqlite3.Row
        rows = [dict(r) for r in con.execute(sql).fetchall()]
        con.close()
    except Exception as e:
        return {"sql": sql, "rows": None, "answer": f"SQL ERROR: {e}"}

    # (4) summarize
    answer = llm.invoke([
        SystemMessage(content=SUMMARY_PROMPT),
        HumanMessage(content=f"Question: {question}\nRows: {rows}"),
    ]).content.strip()
    return {"sql": sql, "rows": rows, "answer": answer}

**6. Detailed walkthrough on one tricky question**

In [30]:
tricky_q = "How many heavy launches have we flown?"
expected_sql = "SELECT COUNT(*) FROM launches WHERE payload_kg > 5000"
expected_answer = "6"

print(f"Question:        {tricky_q}")
print(f"Expected SQL:    {expected_sql}")
print(f"Expected answer: contains '{expected_answer}' (i.e. 6 missions)")

Question:        How many heavy launches have we flown?
Expected SQL:    SELECT COUNT(*) FROM launches WHERE payload_kg > 5000
Expected answer: contains '6' (i.e. 6 missions)


In [31]:
r1 = run_pipeline(prompt_v1, tricky_q) #system_prompt-> prompt_v1
print(f"v1 SQL:    {r1['sql']}")
print(f"v1 rows:   {r1['rows']}")
print(f"v1 answer: {r1['answer']}")
print(f"v1 PASS?   {expected_answer in r1['answer']}")

v1 SQL:    SELECT COUNT(*) FROM launches WHERE vehicle = 'Falcon Heavy'
v1 rows:   [{'COUNT(*)': 2}]
v1 answer: 2
v1 PASS?   False


In [32]:
r2 = run_pipeline(prompt_v2, tricky_q) #system_promt-> prompt_v2
print(f"v2 SQL:    {r2['sql']}")
print(f"v2 rows:   {r2['rows']}")
print(f"v2 answer: {r2['answer']}")
print(f"v2 PASS?   {expected_answer in r2['answer']}")

v2 SQL:    The query you need to run is:

```sql
SELECT COUNT(*) FROM launches WHERE vehicle = 'Falcon Heavy';
```

Executing this statement will return a single integer value—the total number of Falcon Heavy (i.e., “heavy”) launches that have been flown in the `launches` table.
v2 rows:   None
v2 answer: SQL ERROR: near "The": syntax error
v2 PASS?   False


In [33]:
# Unable to run query with Prompt-3 because i am using free Groq Model
# First run-> APIStatusError  Second run-> SQL ERROR: no such column: launch_type
r3 = run_pipeline(prompt_v3, tricky_q) #system_prompt-> prompt_v3
print(f"v3 SQL:    {r3['sql']}")
print(f"v3 rows:   {r3['rows']}")
print(f"v3 answer: {r3['answer']}")
print(f"v3 PASS?   {expected_answer in r3['answer']}")


'''1. Unverified Test Suite->Fail not Pass because of Groq Model, less rows of schema.md taken
   2. Now Pass because Test Suite carefully tested on SQLite (Not a Groq Model Prob)
   3. Sometimes it runs sometimes not Rate limit reached for model `openai/gpt-oss-120b
'''

v3 SQL:    SELECT COUNT(*) AS n FROM launches WHERE payload_kg > 5000;
v3 rows:   [{'n': 6}]
v3 answer: 6
v3 PASS?   True


'1. Unverified Test Suite->Fail not Pass because of Groq Model, less rows of schema.md taken\n   2. Now Pass because Test Suite carefully tested on SQLite (Not a Groq Model Prob)\n   3. Sometimes it runs sometimes not Rate limit reached for model `openai/gpt-oss-120b\n'

**Note**: First run when **unverified Test Suite** APIStatusError: Error code: 413 - {'error': {'message': 'Request Entity Too Large', 'type': 'invalid_request_error', 'code': 'request_too_large'}}

Second Run-> Pass when **Verified Test Suite**

**Conclusion**: Schema Doc and Verified Test Suite both matter ( query failed with prompt-1,2 but passed with prompt-3)

**7. Run the full test suite across all three prompts**
evaluation system for your SQL/LLM pipeline.
It checks whether your generated answers are correct for different prompts (v1, v2, v3).

In [34]:
#APIStatusError: Error code: 413 - {'error': {'message': 'Request Entity Too Large', 'type': 'invalid_request_error', 'code': 'request_too_large'}}
'''
def grade(prompt_label, prompt, suite):
    rows = []
    for case in suite:
        r = run_pipeline(prompt, case["question"])
        passed = case["expected_answer"].lower() in r["answer"].lower()
        rows.append({
            "prompt": prompt_label,
            "question": case["question"][:45] + ("..." if len(case["question"]) > 45 else ""),
            "expected": case["expected_answer"],
            "sql_generated": " ".join(r["sql"].split())[:80] + ("..." if len(r["sql"]) > 80 else ""),
            "answer": r["answer"][:70] + ("..." if len(r["answer"]) > 70 else ""),
            "pass": "PASS" if passed else "FAIL",
        })
    return rows


grades_v1 = grade("v1: DDL only",       prompt_v1, test_suite)
grades_v2 = grade("v2: + descriptions", prompt_v2, test_suite)
grades_v3 = grade("v3: + few-shot",     prompt_v3, test_suite)

all_grades = pd.DataFrame(grades_v1 + grades_v2 + grades_v3)
all_grades'''

'\ndef grade(prompt_label, prompt, suite):\n    rows = []\n    for case in suite:\n        r = run_pipeline(prompt, case["question"])\n        passed = case["expected_answer"].lower() in r["answer"].lower()\n        rows.append({\n            "prompt": prompt_label,\n            "question": case["question"][:45] + ("..." if len(case["question"]) > 45 else ""),\n            "expected": case["expected_answer"],\n            "sql_generated": " ".join(r["sql"].split())[:80] + ("..." if len(r["sql"]) > 80 else ""),\n            "answer": r["answer"][:70] + ("..." if len(r["answer"]) > 70 else ""),\n            "pass": "PASS" if passed else "FAIL",\n        })\n    return rows\n\n\ngrades_v1 = grade("v1: DDL only",       prompt_v1, test_suite)\ngrades_v2 = grade("v2: + descriptions", prompt_v2, test_suite)\ngrades_v3 = grade("v3: + few-shot",     prompt_v3, test_suite)\n\nall_grades = pd.DataFrame(grades_v1 + grades_v2 + grades_v3)\nall_grades'

**Token Count Management**---------Not Proper

In [35]:
print(len(prompt_v1)),print(len(prompt_v2)),print(len(prompt_v3)),print(len(str(test_suite)))   # length is not token ( in starting i am counting this tokens)

980
2025
5427
4653


(None, None, None, None)

In [36]:
# Error with Groq
# best token count comes from the same model/provider you are actually
'''
import tiktoken
enc = tiktoken.get_encoding("groq/compound") #tiktoken does not recognize Groq model names directly.
tokens = enc.encode(prompt_v3)
print(len(tokens))'''

'\nimport tiktoken\nenc = tiktoken.get_encoding("groq/compound") #tiktoken does not recognize Groq model names directly.\ntokens = enc.encode(prompt_v3)\nprint(len(tokens))'

In [37]:
#Solution
#Use approximate tokenizer estimation
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")

# Prompt tokens
v1_tokens = len(enc.encode(prompt_v1))
v2_tokens = len(enc.encode(prompt_v2))
v3_tokens = len(enc.encode(prompt_v3))

# Test suite tokens
test_suite_tokens = len(enc.encode(str(test_suite)))

# Total
grand_total = (v1_tokens +v2_tokens +v3_tokens +test_suite_tokens)

print("Prompt V1:", v1_tokens)
print("Prompt V2:", v2_tokens)
print("Prompt V3:", v3_tokens)
print("Test Suite:", test_suite_tokens)
print("Grand Total:", grand_total)

Prompt V1: 211
Prompt V2: 477
Prompt V3: 1311
Test Suite: 1138
Grand Total: 3137


**Note:** model already consumes about 3137 input tokens before generating SQL/output.
Groq groq/compound has a context window of 131,072 tokens (131K)
131072 - 3137 ≈ 127935 unused tokens


In [38]:
#evaluation pipeline also tracks token usage properly----removed now

In [39]:
small_suite = test_suite[:5]

def grade(prompt_label, prompt, small_suite):
    rows = []
    for case in small_suite:
        r = run_pipeline(prompt, case["question"])
        passed = case["expected_answer"].lower() in r["answer"].lower()
        rows.append({
            "prompt": prompt_label,
            "question": case["question"][:45] + ("..." if len(case["question"]) > 45 else ""),
            "expected": case["expected_answer"],
            "sql_generated": " ".join(r["sql"].split())[:80] + ("..." if len(r["sql"]) > 80 else ""),
            "answer": r["answer"][:70] + ("..." if len(r["answer"]) > 70 else ""),
            "pass": "PASS" if passed else "FAIL",
        })
    return rows


In [40]:
# RUN EVALUATIONS-----15 May TPM Error
'''import time

grades_v1 = grade( "v1: DDL only", prompt_v1,
    #small_suite   #error tokens per minute (TPM): Limit 8000, Used 5001, Requested 6649
    small_suite [:5]
)
pd.DataFrame(grades_v1)'''

'import time\n\ngrades_v1 = grade( "v1: DDL only", prompt_v1,\n    #evaluation_suite   #error tokens per minute (TPM): Limit 8000, Used 5001, Requested 6649\n    small_suite [:5]\n)\npd.DataFrame(grades_v1)'

In [ ]:
# 16 May-----------413 Request Too Large------------One request is too big
#One request is too big
#APIStatusError
import time

grades_v1 = grade(
    "v1: DDL only",
    prompt_v1,
    small_suite[:5]  # 5 test cases
)

pd.DataFrame(grades_v1)

In [ ]:
# RUN EVALUATIONS  small_suite means Evaluation_Suite
'''
import time
grades_v1 = grade( "v1: DDL only", prompt_v1,
    #small_suite   #error tokens per minute (TPM): Limit 8000, Used 5001, Requested 6649
    small_suite[:2]
)
pd.DataFrame(grades_v1)'''

In [ ]:
grades_v2 = grade(
    "v2: + descriptions",
    prompt_v2,
    #small_suite[:2] #tokens per minute (TPM): Limit 8000, Used 3805, Requested 7290.
    small_suite[:1] #2 test cases only
)
time.sleep(5) #can do 30
pd.DataFrame(grades_v2)

In [ ]:
grades_v3 = grade(
    "v3: + few-shot",
    prompt_v3,
    small_suite[:1] #tokens per minute (TPM): Limit 8000, Used 5466, Requested 6514. Please try again in 29.85s.
)
time.sleep(30)
pd.DataFrame(grades_v3)  # after 30 seconds

'''"
15 May,
category": "sorting", this is failed why Few-shot examples changed model behavior Few-shot examples biased output model learned explanatory style OMG

its not running with second code->tokens per day (TPD): Limit 500000, Used 499597, Requested 1416. Please try again in 2m55.0464s.
499597 + 1416 > 500000  That is why request failed.

16 May- this error, biasing concept on 15th may -> how i discovered don't know
APIStatusError: Error code: 413 - {'error': {'message': 'Request Entity Too Large', 'type': 'invalid_request_error', 'code': 'request_too_large'}}

'''

In [ ]:
# COMBINE RESULTS
all_grades = pd.DataFrame(
    grades_v1 + grades_v2 + grades_v3 #NameError: name 'grades_v1' is not defined
)


In [ ]:
# FINAL OUTPUT
all_grades #error

In [ ]:
# TOKEN SUMMARY #error
print(
    "Total Input Tokens:",
    all_grades["input_tokens"].sum()
)
print(
    "Total Output Tokens:",
    all_grades["output_tokens"].sum()
)
print(
    "Total Tokens:",
    all_grades["total_tokens"].sum()
)


In [ ]:
# A compact pass-rate summary #NameError: name 'grades_v1' is not defined
summary = pd.DataFrame({
    "question": [c["question"][:40] + ("..." if len(c["question"]) > 40 else "") for c in test_suite],
    "expected": [c["expected_answer"] for c in test_suite],
    "v1: DDL only":       [g["pass"] for g in grades_v1],
    "v2: + descriptions": [g["pass"] for g in grades_v2],
    "v3: + few-shot":     [g["pass"] for g in grades_v3],
})

print(f"v1 (DDL only):           {sum(1 for g in grades_v1 if g['pass']=='PASS')} / {len(test_suite)}")
print(f"v2 (+ descriptions):     {sum(1 for g in grades_v2 if g['pass']=='PASS')} / {len(test_suite)}")
print(f"v3 (+ few-shot):         {sum(1 for g in grades_v3 if g['pass']=='PASS')} / {len(test_suite)}")
print(summary)

**Full pipeline walkthrough on a single question (using v3)**

In [58]:
# (1) the user's question
question = "Which vehicle has the highest success rate?"
print(f"USER QUESTION:\n  {question}\n")

USER QUESTION:
  Which vehicle has the highest success rate?



In [59]:
# (2) ask the LLM to write the SQL
sql_response = llm.invoke([
    SystemMessage(content=prompt_v3),
    HumanMessage(content=question),
])
sql = sql_response.content.strip()
if sql.startswith("```"):
    sql = sql.split("```")[1]
    if sql.lstrip().lower().startswith("sql"):
        sql = sql.lstrip()[3:]
    sql = sql.strip()

print(f"GENERATED SQL:\n{sql}\n")

GENERATED SQL:
SELECT vehicle,
       ROUND(100.0 * SUM(success) / COUNT(*), 1) AS success_pct
FROM launches
GROUP BY vehicle
ORDER BY success_pct DESC
LIMIT 1



In [60]:
# (3) validate
upper = sql.upper()
for w in FORBIDDEN:
    assert w not in upper, f"refused: {w}"
assert "FROM LAUNCHES" in upper.replace("\n", " "), "refused: must read from launches"
aggs = ("COUNT(", "SUM(", "AVG(", "MIN(", "MAX(", "GROUP BY")
if "LIMIT" not in upper and not any(a in upper for a in aggs):
    sql = sql.rstrip(";\n ") + " LIMIT 100"

print(f"VALIDATED SQL:\n{sql}\n")

VALIDATED SQL:
SELECT vehicle,
       ROUND(100.0 * SUM(success) / COUNT(*), 1) AS success_pct
FROM launches
GROUP BY vehicle
ORDER BY success_pct DESC
LIMIT 1



In [61]:
# (4) execute on SQLite
con = sqlite3.connect(DB)
con.row_factory = sqlite3.Row
rows = [dict(r) for r in con.execute(sql).fetchall()]
con.close()

print("ROWS FROM DB:")
for r in rows:
    print(" ", r)

ROWS FROM DB:
  {'vehicle': 'Falcon Heavy', 'success_pct': 100.0}


In [ ]:
# (5) ask the LLM to write the English answer
answer_response = llm.invoke([
    SystemMessage(content=SUMMARY_PROMPT),
    HumanMessage(content=f"Question: {question}\nRows: {rows}"),
])
answer = answer_response.content.strip()

print(f"FINAL ANSWER:\n  {answer}")

#(TPM): Limit 8000, Used 4716, Requested 6446. Please try again in 23.715s. Need more tokens?

#Three failure modes worth seeing
**Failure 1: validator catches dangerous SQL**

In [63]:
r = run_pipeline(prompt_v3, "Delete all the failed launches")
print(f"SQL:    {r['sql']}")
print(f"Answer: {r['answer']}")

SQL:    SELECT launch_id FROM launches WHERE success = 0 LIMIT 100
Answer: I couldn't find an answer.


**Failure 2: question is out of scope (the database has no answer)**

In [ ]:
r = run_pipeline(prompt_v3, "What year did SpaceX go public?")
print(f"SQL:    {r['sql']}")
print(f"Answer: {r['answer']}")
'''APIStatusError: Error code: 413 - {'error': {'message': 'Request Entity Too Large''''

**Failure 3: ambiguity is interpreted, not refused**

In [ ]:
r = run_pipeline(prompt_v3, "How many Falcon X missions have flown?")
print(f"SQL:    {r['sql']}")
print(f"Answer: {r['answer']}")
'''Request Entity Too Large'''

**Code-2 Full Text2SQL Pipeline (Multiple Functions)**

In [68]:
#system_prompt = prompt_v3

# SUMMARY PROMPT
SUMMARY_PROMPT1 = (
    "Answer the user's question in one short sentence using only the rows. "
    "Use digits (e.g. '14', not 'fourteen') and cite numbers exactly. "
    "Do not speculate. "
    "If rows is empty, say you couldn't find an answer."
)

# FORBIDDEN SQL WORDS
FORBIDDEN = ("INSERT", "UPDATE", "DELETE", "DROP", "ALTER", "TRUNCATE")

# 1. GENERATE SQL
def generate_sql(question):
    response = llm.invoke([
        SystemMessage(content=prompt_v3),
        HumanMessage(content=question)
    ])
    sql = response.content.strip()
    # Remove markdown if model returns ```sql
    if sql.startswith("```"):
        sql = sql.replace("```sql", "").replace("```", "").strip()
    return sql

# 2. VALIDATE SQL
def validate_sql(sql):
    upper = sql.upper()
    # Block dangerous SQL
    for word in FORBIDDEN:
        if word in upper:
            return False, f"REFUSED: contains {word}"
    # Allow only launches table
    if "FROM LAUNCHES" not in upper.replace("\n", " "):
        return False, "REFUSED: must use launches table"
    # Aggregation keywords
    aggs = ("COUNT(", "SUM(", "AVG(", "MIN(", "MAX(", "GROUP BY")
    # Auto add LIMIT
    if "LIMIT" not in upper and not any(a in upper for a in aggs):
        sql = sql.rstrip("; \n") + " LIMIT 100"
    return True, sql

# 3. RUN SQL
def run_sql(sql):
    try:
        conn = sqlite3.connect(DB)
        conn.row_factory = sqlite3.Row
        rows = [dict(r) for r in conn.execute(sql).fetchall()]
        conn.close()
        return rows
    except Exception as e:
        return f"SQL ERROR: {e}"

# 4. SUMMARIZE ANSWER
def summarize_answer(question, rows):
    rows = rows[:10]  # test-> error bcoz of Groq Model so sending less rows
    response = llm.invoke([
        SystemMessage(content=SUMMARY_PROMPT1),
        HumanMessage(content=f"Question:\n{question}\n\nRows:\n{rows}")
    ])
    return response.content.strip()

# 5. MAIN PIPELINE
def run_pipeline1(question):

    # Generate SQL
    sql = generate_sql(question)

    # Validate SQL
    valid, result = validate_sql(sql)
    if not valid:
        return {"sql": sql, "rows": None, "answer": result}
    sql = result
    # Run SQL
    rows = run_sql(sql)
    if isinstance(rows, str):
        return {"sql": sql, "rows": None, "answer": rows}

    # Summarize
    answer = summarize_answer(question, rows)

    # Final Output
    return {"sql": sql, "rows": rows, "answer": answer}

In [69]:
# TEST
question = "How many Falcon 9 launches failed?"
result = run_pipeline1(question)

print(f"Question: {question}")
print(f"Generated SQL: {result['sql']}")
print(f"Final Answer: {result['answer']}")
print(f"Rows: {result['rows']}")

Question: How many Falcon 9 launches failed?
Generated SQL: SELECT COUNT(*) AS n FROM launches WHERE vehicle = 'Falcon 9' AND success = 0;
Final Answer: 1
Rows: [{'n': 1}]


In [ ]:
# deleted code on 16th may , previously i got schema grounding problem on 15th May

**Note:** schema grounding problem  in GROQ

In [50]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="groq/compound",
    temperature=0
)

In [72]:
import sqlite3

conn = sqlite3.connect(DB)

cursor = conn.cursor()

cursor.execute("""
SELECT name
FROM sqlite_master
WHERE type='table'
""")

print(cursor.fetchall())

conn.close()

[('launches',)]


**SELF-HEALING SQL REPAIR**

In [73]:
# SUMMARY PROMPT
SUMMARY_PROMPT = (
    "Answer the user's question in one short sentence using only the rows. "
    "Use digits exactly as given. "
    "Do not speculate. "
    "If rows are empty, say you could not find an answer."
)

# FORBIDDEN SQL
FORBIDDEN = (
    "INSERT",
    "UPDATE",
    "DELETE",
    "DROP",
    "ALTER",
    "TRUNCATE"
)

# AUTO SCHEMA EXTRACTION Agent automatically reads SQLite/database structure (tables, columns, DDL)

def build_schema_context():
    conn = sqlite3.connect(DB)
    cursor = conn.cursor()
    cursor.execute("PRAGMA table_info(launches)")
    columns = cursor.fetchall()
    conn.close()
    schema = "Table: launches\n\nColumns:\n"
    for col in columns:
        col_name = col[1]
        col_type = col[2]
        schema += f"- {col_name} ({col_type})\n"
    # Optional semantic grounding
    schema += """
Semantic Meanings:
- vehicle = rocket name
- success = 1 means success, 0 means failure
- payload_kg = payload mass in kilograms
"""
    return schema

SCHEMA_CONTEXT = build_schema_context()

# GENERATE SQL
def generate_sql(question):
    grounded_prompt = f"""
{prompt_v3}

Database Schema:
{SCHEMA_CONTEXT}
"""
    response = llm.invoke([
        SystemMessage(content=grounded_prompt),
        HumanMessage(content=question)
    ])
    sql = response.content.strip()
    # Remove markdown
    if sql.startswith("```"):
        sql = sql.replace("```sql", "")
        sql = sql.replace("```", "")
        sql = sql.strip()
    return sql

# VALIDATE SQL
def validate_sql(sql):
    upper = sql.upper()
    # Block dangerous SQL
    for word in FORBIDDEN:
        if word in upper:
            return False, f"REFUSED: contains {word}"
    # Allow only launches table
    if "FROM LAUNCHES" not in upper.replace("\n", " "):
        return False, "REFUSED: must use launches table"
    # Aggregation keywords
    aggs = (
        "COUNT(",
        "SUM(",
        "AVG(",
        "MIN(",
        "MAX(",
        "GROUP BY"
    )
    # Auto add LIMIT
    if "LIMIT" not in upper and not any(a in upper for a in aggs):
        sql = sql.rstrip("; \n") + " LIMIT 100"
    return True, sql

# RUN SQL
def run_sql(sql):
    try:
        conn = sqlite3.connect(DB)
        conn.row_factory = sqlite3.Row
        rows = [
            dict(r)
            for r in conn.execute(sql).fetchall()
        ]
        conn.close()
        return rows
    except Exception as e:
        return f"SQL ERROR: {e}"

# SELF-HEALING SQL REPAIR
def repair_sql(question, failed_sql, error_message):
    repair_prompt = f"""
The previous SQL query failed.

Database Schema:
{SCHEMA_CONTEXT}

User Question:
{question}

Failed SQL:
{failed_sql}

SQL Error:
{error_message}

Instructions:
- Fix the SQL query
- Use ONLY launches table
- Return ONLY corrected SQL
- No explanation
"""
    response = llm.invoke([
        SystemMessage(content="You are a SQL repair agent."),
        HumanMessage(content=repair_prompt)
    ])
    sql = response.content.strip()
    # Remove markdown
    if sql.startswith("```"):
        sql = sql.replace("```sql", "")
        sql = sql.replace("```", "")
        sql = sql.strip()
    return sql

# SUMMARIZE ANSWER
def summarize_answer(question, rows):
    # Reduce rows for smaller models
    rows = rows[:10]
    response = llm.invoke([
        SystemMessage(content=SUMMARY_PROMPT),
        HumanMessage(
            content=f"""
Question:
{question}

Rows:
{rows}
"""
        )
    ])
    answer = response.content.strip()
    return answer

# MAIN PIPELINE

def run_pipeline4(question):
    # STEP 1: Generate SQL
    sql = generate_sql(question)
    print("\nGENERATED SQL:")
    print(sql)

    # STEP 2: Validate SQL
    valid, result = validate_sql(sql)
    if not valid:
        return {"sql": sql, "rows": None, "answer": result}
    sql = result

    # STEP 3: Self-Healing Execution
    max_retries = 3
    for attempt in range(max_retries):
        print(f"\nATTEMPT {attempt + 1}")
        rows = run_sql(sql)
        if not isinstance(rows, str):
            print("SQL SUCCESS")
            break
        error_message = rows
        print("SQL FAILED:")
        print(error_message)
        sql = repair_sql(question=question, failed_sql=sql, error_message=error_message)
        print("\nREPAIRED SQL:")
        print(sql)
        valid, result = validate_sql(sql)
        if not valid:
            return {"sql": sql, "rows": None, "answer": result}
        sql = result

    # STEP 4: Final Failure
    if isinstance(rows, str):
        return {"sql": sql, "rows": None, "answer": rows}

    # STEP 5: Summarize
    answer = summarize_answer(question, rows)

    # FINAL OUTPUT
    return {"sql": sql, "rows": rows, "answer": answer}

In [74]:
# TEST

question = "How many Falcon 9 launches failed?"
result = run_pipeline4(question)

print(f"Question: {question}")
print(f"Generated SQL: {result['sql']}")
print(f"Final Answer: {result['answer']}")
print(f"Rows: {result['rows']}")


GENERATED SQL:
SELECT COUNT(*) AS n FROM launches WHERE vehicle = 'Falcon 9' AND success = 0;

ATTEMPT 1
SQL SUCCESS
Question: How many Falcon 9 launches failed?
Generated SQL: SELECT COUNT(*) AS n FROM launches WHERE vehicle = 'Falcon 9' AND success = 0;
Final Answer: 1
Rows: [{'n': 1}]


In [75]:
question = "How many heavy launches have we flown?"
result = run_pipeline4(question)

print(f"Question: {question}")
print(f"Generated SQL: {result['sql']}")
print(f"Final Answer: {result['answer']}")
print(f"Rows: {result['rows']}")


GENERATED SQL:
SELECT COUNT(*) AS n FROM launches WHERE payload_kg > 5000;

ATTEMPT 1
SQL SUCCESS
Question: How many heavy launches have we flown?
Generated SQL: SELECT COUNT(*) AS n FROM launches WHERE payload_kg > 5000;
Final Answer: 6
Rows: [{'n': 6}]
